# Nigeria 2030: financing, electricity mix and policy

This notebook reads **revised solved networks**, never the old metrics cache. Run cells in order.

**Main question:** how does renewable financing affect the least-cost 2030 electricity mix when gas financing is held fixed?

Rates on charts are real rates. The nominal source rates and inflation assumption remain in each network's metadata. Policy cases are conditional financing experiments, not measured policy effects.

Results use real 2024 USD. Primary generation excludes battery discharge and load shedding. Hydro discharge counts as primary generation only when it is not charged from the grid.

This local-code update retains the existing/new conventional-capacity split, the common 14.68% nominal-equivalent transmission rate and the numerical unmet-demand penalty of 10,000 per MWh in the revised USD accounting. The penalty is a modelling assumption, not a measured Nigerian value of lost load.

Cost-file correction: the source snapshot contains some 2013 EUR costs, which are explicitly rebased before currency conversion. Utility-scale solar uses no rooftop blend. Gas quotes are in 2026 USD/MMBtu; model fuel costs are in 2024 USD/MWh_fuel. See COST_REVIEW.md.

The existing renewable fleet is retained. The 8,784 weather snapshots carry a total annual weight of 8,760 hours; annualised costs use that same represented-year convention.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "Snakefile").exists():
    ROOT = ROOT.parent
assert (ROOT / "Snakefile").exists(), "Open this notebook from the repository or analysis folder"
sys.path.insert(0, str(ROOT / "analysis"))
from nigeria_results import build_results, plot_results

ALLOW_PARTIAL = False  # True ONLY for preliminary inspection after the first B2 run
OUTPUT = ROOT / "analysis/output/nigeria_consistent"
df = build_results(ROOT, allow_partial=ALLOW_PARTIAL)
print(f"Loaded {len(df)} solved scenarios. Partial mode: {ALLOW_PARTIAL}")

## 1. Check the evidence before interpreting the charts

The extractor checks the emissions cap and reconciles the objective with investment and operating costs. Inspect unmet demand: a low-cost scenario with material unmet demand is not an adequate supply solution.

The main connected component represents the full 78 TWh demand. Excluded locations are not claimed to have been physically connected. Review `nigeria_audit` in the baseline network for the removed input load and rescaling factor.

In [ ]:
columns = ["scenario", "rate_basis", "renewable_wacc", "gas_wacc", "demand_twh", "served_twh", "unserved_share", "emissions_mt", "cap_mt", "objective_residual_usd"]
display(df[columns].round(5))
if (df.unserved_share > 0.001).any():
    print("REVIEW: more than 0.1% of demand is unserved in at least one case. This is a reporting flag, not an official reliability standard.")
import pypsa
baseline_path = ROOT / "results/nigeria_consistent_B2/networks/elec_s_10_ec_lcopt_1h.nc"
if baseline_path.exists():
    baseline = pypsa.Network(baseline_path)
    display(baseline.meta.get("nigeria_audit", {}))
    display(baseline.generators.loc[baseline.generators.carrier.isin(["CCGT", "OCGT"]), ["carrier", "p_nom_extendable", "p_nom_opt", "p_max_pu", "efficiency", "marginal_cost"]])

## 2. Core results: financing changes investment and generation

Figure 1 shows new investment and primary generation across the WACC sweep. Figure 2 shows renewable share, annual planning supply cost and operational emissions. Gas financing is fixed within the core sweep; transmission financing is also fixed.

The 50% line is a national policy benchmark; this is a grid model. The 8.8 Mt line is an analytical allocation, not an official grid emissions target. Do not claim exact switching thresholds between sampled WACC values.

In [ ]:
figures = plot_results(df, OUTPUT / "figures")
plt.show()
print("Saved PNG and PDF figures to", OUTPUT / "figures")

## 3. Compare policy cases with the same reference

Use B2 as the reference for P1/P1D and P2–P4. D2 imposes the analytical emissions cap. P1 changes the gas-price quote on the same 2026 price basis; P2–P4 change assumed financing conditions. The assumed nominal reductions of 2 and 4 percentage points are converted consistently to real rates.

B1/B3 change gas financing too and are not part of the renewable-only sweep. The battery sensitivity separates battery financing from renewable-generation financing.

In [ ]:
if "B2" in set(df.scenario):
    base = df.set_index("scenario").loc["B2"]
    comparison = df.loc[df.scenario.isin(["B2", "D2", "P1", "P1D", "P2", "P3", "P4", "S2a", "S2b"])].copy()
    comparison["cost_change_pct"] = 100 * (comparison.planning_supply_cost_usd/base.planning_supply_cost_usd-1)
    comparison["emissions_change_mt"] = comparison.emissions_mt-base.emissions_mt
    comparison["renewable_change_pp"] = 100*(comparison.renewable_share-base.renewable_share)
    display(comparison[["scenario", "cost_change_pct", "emissions_change_mt", "renewable_change_pp", "unserved_share"]].round(3))
    comparison.to_csv(OUTPUT / "policy_comparison.csv", index=False)

## 4. Cost definitions and flexibility

**Planning supply cost** is the objective excluding the unmet-demand penalty. It contains investment in additional/extendable capacity and operating costs; it is not a full replacement cost for the existing fleet.

**System LCOE** here uses the equivalent annual cost of the complete modelled stock plus operation, excluding the unmet-demand penalty, divided by primary generation. Existing assets are valued at replacement-cost annuities. Fixed hydro coefficients are reconstructed from the reviewed technology-cost input because the optimisation network stores zero capital cost for those fixed assets. This is an accounting convention, not historical cash expenditure. It includes battery and transmission costs; fixed O&M already in `capital_cost` is not added twice.

Both measures should be accompanied by unmet demand. Use annual costs rather than LCOE alone as the main policy comparison.

In [ ]:
display(df[["scenario", "planning_supply_cost_usd", "planning_cost_per_served_mwh", "system_lcoe_usd_mwh", "battery_energy_gwh", "battery_discharge_twh", "line_expansion_gw_km", "vre_curtailment_share", "new_CCGT_utilisation", "new_OCGT_utilisation"]].round(3))

## 5. Robustness and limits of interpretation

Compare B2 with S_nominal (original nominal-rate stress convention), S_inflation4 (4% assumed USD inflation) and S_gas85 (85% new-gas availability). These are explicit sensitivities, not observations.

Retain the committed hydro normalization/multiplier for this version. Do not claim that the additional 0.476/0.756 correction has been applied or that inflows were independently validated. Confirm hydro generation after the first rerun.

A 2030 model can discuss new gas exposure and investment preparation for further decarbonisation. It cannot establish a 2040/2060 pathway or prove that assets become stranded.

In [ ]:
display(df.loc[df.scenario.isin(["B2", "S_nominal", "S_inflation4", "S_gas85"]), ["scenario", "renewable_wacc", "gas_wacc", "renewable_share", "emissions_mt", "planning_supply_cost_usd", "unserved_share"]].round(4))